In [9]:
pip install langchain langchain-openai openpyxl

In [10]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from google.colab import userdata

LangChain is a Python framework that helps connect an LLM to prompts, data, and tools.

## Create LLM

In [11]:
llm = None
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        api_key=userdata.get("OPENAI_APIKEY"),
        temperature=0
    )
except Exception as e:
    print(f"Error creating model: {e}")

## Read Input csv files and store in dataframe

In [12]:
import pandas as pd

In [29]:
credit_card_terms_df = pd.read_csv('/content/credit_card_terms.csv')
credit_card_terms_xlsx_df = pd.read_excel('/content/credit_card_terms.xlsx')
ecommerce_faqs_df = pd.read_csv('/content/ecommerce_faqs.csv')
hospital_policy_df = pd.read_csv('/content/hospital_policy.csv')
saas_docs_df = pd.read_csv('/content/saas_docs.csv')

saas_docs_df.head()

,Doc ID,Feature,Plan Level,Description,Technical Limit,Related API
0,S-501,API Rate Limit,Free,"Users on the Free tier are limited to 1,000 AP...",1000 req/day,GET /v1/data
1,S-502,API Rate Limit,Enterprise,Enterprise workspaces have a dedicated pool of...,1M req/day,ALL
2,S-503,User Roles,All,Admins have full access to billing and workspa...,NaN,POST /v1/users
3,S-504,Data Export,Pro,"Data can be exported in CSV, JSON, or XML form...",5GB max,GET /v1/export
4,S-505,Integrations,Pro,Slack integration allows real-time notificatio...,5 channels,Webhooks


## Create a tool which the agent can use.

Note the tool description is required and the `@tool` decorator is what makes this a tool.

In [30]:
dataframes = {
    "credit_card_terms": credit_card_terms_df,
    "credit_card_terms_xlsx": credit_card_terms_xlsx_df,
    "ecommerce_faqs": ecommerce_faqs_df,
    "hospital_policy": hospital_policy_df,
    "saas_docs": saas_docs_df
}

In [31]:
@tool
def read_data(name: str) -> str:
    """Read one dataframe using its exact name."""

    print(f"Inspecting the dataframe: {name}")

    if name not in dataframes:
        return (
            f"Dataframe '{name}' was not found. "
            f"Available dataframes: {list(dataframes.keys())}"
        )

    return dataframes[name].to_json(orient="records")

## Create Agent

### Prompting tips

Use this simple structure:
* Role — who the agent is
* Task — what it must do
* Data/tools — what it can use
* Rules — what it must not do
* Fallback — what to say when it cannot answer
* Output — how the answer should look

In [36]:
dataframe_names = ", ".join(dataframes.keys())

system_prompt = (
    "You are a support assistant. "
    "Answer questions only using the available DataFrames. "
    f"The valid DataFrame names are: {dataframe_names}. "
    "When calling read_data, use exactly one of these names. "
    "Do not invent or modify a DataFrame name. "
    "Choose the most relevant DataFrame first. "
    "Do not use general knowledge. "
    "If the answer is not present in the data, clearly say it was not found. "
    "Return a concise answer."
)

In [38]:
agent = create_agent(
    model=llm,
    tools=[read_data],
    system_prompt=system_prompt
)

## Invoking Agent

In [39]:
def run_agent(prompt: str):
  """
  The main execution loop for the CSV FAQ Agent.
  """
  print(f"\n--- Running the CSV FAQ Agent ---")

  try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    return result["messages"][-1].content

  except Exception as e:
    return f"Agent error: {e}"


## Calling the Agent to test

### Credit Card

In [40]:
answer = run_agent("What is the annual fee for the Platinum Rewards card?")
print(answer)


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: credit_card_terms_xlsx
Inspecting the dataframe: credit_card_terms
The annual fee for the Platinum Rewards card is $495.


In [42]:
run_agent("How many points do I earn on flights with the Platinum Rewards card?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: credit_card_terms
Inspecting the dataframe: credit_card_terms_xlsx


'With the Platinum Rewards card, you earn 5 points per $1 spent on flights booked directly with airlines or via the travel portal.'

In [41]:
run_agent("What is the capital of France?")


--- Running the CSV FAQ Agent ---


'The answer was not found in the data.'

### E Commerce

In [43]:
run_agent("How long does standard shipping take?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: ecommerce_faqs


'Standard shipping typically takes 3-5 business days for domestic orders. For rural areas, please allow up to 7 business days. International shipping generally takes 10-15 business days, depending on customs processing.'

In [44]:
run_agent("What should I do if my item arrives damaged?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: ecommerce_faqs


'If your item arrives damaged, contact support within 48 hours with photos of the box and item. A replacement will be shipped immediately without waiting for the return.'

### Hospital

In [45]:
run_agent("How can I request my medical records?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: hospital_policy


'You can request your medical records via the patient portal or by submitting Form MR-99 in person. Processing takes up to 14 business days, and there is a fee of $0.10 per page for printed copies.'

In [46]:
run_agent("What are the general visiting hours?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: hospital_policy


'General visiting hours are from 10:00 AM to 8:00 PM. A maximum of 2 visitors per patient is allowed at a time, and children under 12 are not permitted in ICU or infectious disease wards.'

In [47]:
run_agent("Can a parent stay overnight with a pediatric patient?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: hospital_policy


'Yes, one parent or legal guardian is permitted to stay overnight with pediatric patients. A sleeper chair is provided in the room. Siblings are not permitted to stay overnight.'

### SaaS

In [48]:
run_agent("How many API requests can a Free plan user make per day?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: saas_docs


'A Free plan user can make 1,000 API requests per day.'

In [49]:
run_agent("How long are audit logs retained?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: saas_docsInspecting the dataframe: ecommerce_faqs



'Audit logs are retained for 7 years to comply with regulatory standards.'

### Not found

In [50]:
run_agent("Does the hospital provide free Wi-Fi?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: hospital_policy


'The information about whether the hospital provides free Wi-Fi was not found in the hospital policy data.'

In [51]:
run_agent("Does the SaaS product integrate with Salesforce?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: saas_docs


'The SaaS product documentation does not mention an integration with Salesforce.'

In [52]:
run_agent("Can I use the Platinum Rewards card at Costco?")


--- Running the CSV FAQ Agent ---
Inspecting the dataframe: credit_card_terms
Inspecting the dataframe: credit_card_terms_xlsx


'The data does not specify whether the Platinum Rewards card can be used at Costco.'